# RQ2 behavioral causal pathways

**Research question:** Which behavioral pathways among engagement, assessment activity, pacing, deadlines, and forum interaction have the strongest causal effect on dropout and failure risk in online education?

This notebook is designed for Kaggle execution and saves all result tables as CSV files and figures as PDF files under `/kaggle/working/results/rq2_behavioral_causal_pathways`.

In [1]:
import os, sys, json, math, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path('/kaggle/input/datasets/kimdaligermany/seoyeon-thesis-src')
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.config import ExperimentConfig
from src.data_utils import load_generic_education_dataset, build_weekly_timelines, derive_targets, cumulative_snapshot, make_sequence_tensor
from src.feature_engineering import numeric_feature_columns, make_preprocessor
from src.models import fit_dual_tabular_model, predict_dual_tabular_model, LSTMClassifier, TransformerClassifier, MultiTaskCausalNet, train_torch_model, predict_torch_multitask
from src.evaluation import classification_summary, summarise_dual_task, topk_alert_precision, expected_calibration_error
from src.causal_utils import correlation_dag, direct_indirect_effects, bootstrap_edge_stability
from src.counterfactual_utils import generate_simple_counterfactuals, evaluate_counterfactuals, segment_joint_risk
from src.plotting_utils import lineplot, scatterplot, heatmap, barplot

CFG = ExperimentConfig()
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'

CFG.dataset_name = "rq2_behavioral_causal_pathways"
OUT = CFG.output_dir
print("Output directory:", OUT)

Output directory: /kaggle/working/results/rq2_behavioral_causal_pathways


In [2]:
# Load raw logs, normalize them into a weekly timeline, and derive labels.
raw_df = load_generic_education_dataset(DATASET_PATH, dataset_name=CFG.dataset_name)
weekly_df = build_weekly_timelines(raw_df)
weekly_df = derive_targets(weekly_df)

weekly_df.head()

,student_id,week,code_module,code_presentation,sum_click,n_activities,n_submissions,mean_score,late_rate,gender,age_band,imd_band,highest_education,num_of_prev_attempts,studied_credits,dropout,failure,final_result
0,6516,0,AAA,2014J,485,89,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
1,6516,1,AAA,2014J,42,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
2,6516,2,AAA,2014J,79,20,1.0,0.6,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
3,6516,3,AAA,2014J,193,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
4,6516,4,AAA,2014J,69,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass


## RQ2 workflow

This notebook estimates interpretable causal pathways and decomposes direct versus indirect behavioral effects.

In [3]:
snap = cumulative_snapshot(weekly_df, up_to_week=max(CFG.early_weeks))
feature_cols = numeric_feature_columns(snap)

edges_df = correlation_dag(snap, feature_cols, threshold=0.10)
edges_df.to_csv(OUT / "table_2_1_stable_causal_edges_raw.csv", index=False)

stability_df = bootstrap_edge_stability(snap, feature_cols, n_boot=30)

stable_edges = edges_df.copy()
stable_edges["edge"] = stable_edges["source"].astype(str) + " - " + stable_edges["target"].astype(str)

stable_edges = stable_edges.merge(
    stability_df[["edge", "bootstrap_stability", "stable"]],
    on="edge",
    how="left"
).fillna({"bootstrap_stability": 0.0, "stable": False})

stable_edges = stable_edges.sort_values(
    "bootstrap_stability", ascending=False
).reset_index(drop=True)

stable_edges.to_csv(OUT / "table_2_1_stable_causal_edges_and_effect_strengths.csv", index=False)
stable_edges.head(15)

,source,target,from,to,weight,abs_weight,edge,bootstrap_stability,stable
0,mean_score,dropout,mean_score,dropout,-0.2160,0.2160,mean_score - dropout,1.0000,True
1,n_activities,failure,n_activities,failure,-0.1546,0.1546,n_activities - failure,0.7333,True
2,sum_click,failure,sum_click,failure,-0.1414,0.1414,sum_click - failure,0.1333,False
3,studied_credits,dropout,studied_credits,dropout,0.1405,0.1405,studied_credits - dropout,0.1000,False
4,n_submissions,dropout,n_submissions,dropout,-0.1204,0.1204,n_submissions - dropout,0.0000,False


In [4]:
# Figure 2.1: adjacency heatmap
adj = pd.pivot_table(stable_edges.head(25), index="source", columns="target", values="weight", fill_value=0)
heatmap(adj, title="Figure 2.1 — Learned temporal causal graph", path=OUT / "figure_2_1_learned_temporal_causal_graph.pdf")
adj

target,dropout,failure
source,,
mean_score,-0.2160,0.0000
n_activities,0.0000,-0.1546
n_submissions,-0.1204,0.0000
studied_credits,0.1405,0.0000
sum_click,0.0000,-0.1414


In [5]:
effects_dropout = direct_indirect_effects(snap, feature_cols, "dropout")
effects_failure = direct_indirect_effects(snap, feature_cols, "failure")

effects_dropout.to_csv(OUT / "table_2_2_direct_indirect_total_effects_dropout.csv", index=False)
effects_failure.to_csv(OUT / "table_2_3_direct_indirect_total_effects_failure.csv", index=False)

# A compact combined table aligned with the proposal's expected result format.
combined = effects_dropout.rename(columns={
    "direct_effect":"dropout_direct_effect",
    "indirect_effect":"dropout_indirect_effect",
    "total_effect":"dropout_total_effect",
    "primary_role":"dropout_primary_role"
}).merge(
    effects_failure.rename(columns={
        "direct_effect":"failure_direct_effect",
        "indirect_effect":"failure_indirect_effect",
        "total_effect":"failure_total_effect",
        "primary_role":"failure_primary_role"
    }),
    on="behavior", how="outer"
)
combined.to_csv(OUT / "table_2_4_combined_behavior_effects.csv", index=False)
combined.head(15)

,behavior,dropout_direct_effect,dropout_indirect_effect,dropout_total_effect,dropout_primary_role,failure_direct_effect,failure_indirect_effect,failure_total_effect,failure_primary_role
0,late_rate,-0.0090,-0.8242,-0.8333,Proxy,0.3079,-0.0195,0.2884,Cause
1,mean_score,-0.8833,-0.4235,-1.3068,Cause,-0.2280,0.2856,0.0576,Mixed
2,n_activities,-0.0014,-0.0004,-0.0018,Cause,-0.0047,-0.0003,-0.0050,Cause
3,n_submissions,-0.3190,-0.4904,-0.8094,Proxy,0.1071,-0.0160,0.0911,Cause
4,num_of_prev_attempts,0.0735,-0.0108,0.0627,Cause,0.0939,0.0039,0.0978,Cause
5,studied_credits,0.0016,-0.0002,0.0013,Cause,-0.0001,0.0000,-0.0001,Cause
6,sum_click,-0.0004,-0.0001,-0.0005,Cause,-0.0009,-0.0001,-0.0010,Cause


In [6]:
# Figure 2.2: direct vs indirect pathway decomposition
plot_df = effects_dropout.head(12).copy()
plot_df["abs_direct"] = plot_df["direct_effect"].abs()
plot_df["abs_indirect"] = plot_df["indirect_effect"].abs()

plt.figure(figsize=(9, 5))
x = np.arange(len(plot_df))
plt.bar(x - 0.2, plot_df["abs_direct"], width=0.4, label="Direct")
plt.bar(x + 0.2, plot_df["abs_indirect"], width=0.4, label="Indirect")
plt.xticks(x, plot_df["behavior"], rotation=45, ha="right")
plt.title("Figure 2.2 — Direct and mediated pathway decomposition")
plt.ylabel("Absolute effect")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "figure_2_2_direct_and_mediated_pathway_decomposition.pdf")
plt.close()

plot_df

,behavior,direct_effect,indirect_effect,total_effect,primary_role,abs_direct,abs_indirect
0,mean_score,-0.8833,-0.4235,-1.3068,Cause,0.8833,0.4235
1,late_rate,-0.0090,-0.8242,-0.8333,Proxy,0.0090,0.8242
2,n_submissions,-0.3190,-0.4904,-0.8094,Proxy,0.3190,0.4904
3,num_of_prev_attempts,0.0735,-0.0108,0.0627,Cause,0.0735,0.0108
4,n_activities,-0.0014,-0.0004,-0.0018,Cause,0.0014,0.0004
5,studied_credits,0.0016,-0.0002,0.0013,Cause,0.0016,0.0002
6,sum_click,-0.0004,-0.0001,-0.0005,Cause,0.0004,0.0001
